# GPU-accelerated simulation using Warp

In this tutorial, TODO.

TODO: Explain what Warp is and how GPU computing is in general.

In [ ]:
from flygym.warp.utils import check_gpu

check_gpu()

## Basic GPU simulation

As in _Tutorial 2: Replaying experimental recordings and inferring dynamical quantities_, we start by creating a fly spawned into an empty world:

In [ ]:
import flygym
from flygym.compose import Fly, ActuatorType, FlatGroundWorld
from flygym.anatomy import Skeleton, JointPreset, ActuatedDOFPreset, AxisOrder
from flygym.compose.pose import KinematicPose
from flygym.utils.math import Rotation3D


def make_model(
    joints_preset=JointPreset.LEGS_ONLY,
    actuated_dofs_preset=ActuatedDOFPreset.LEGS_ACTIVE_ONLY,
    actuator_type=ActuatorType.POSITION,
    position_gain=50.0,
    neutral_pose_file=flygym.assets_dir / "model/pose/neutral.yaml",
    spawn_position=(0, 0, 0.7),  # xyz in mm
    spawn_rotation=Rotation3D("quat", (1, 0, 0, 0)),
):

    fly = Fly()
    axis_order = AxisOrder.YAW_PITCH_ROLL

    # Add joints
    skeleton = Skeleton(axis_order=axis_order, joint_preset=joints_preset)
    neutral_pose = KinematicPose(path=neutral_pose_file)
    fly.add_joints(skeleton, neutral_pose=neutral_pose)

    # Add position actuators and set them to the neutral pose
    actuated_dofs_list = fly.skeleton.get_actuated_dofs_from_preset(
        actuated_dofs_preset
    )
    fly.add_actuators(
        actuated_dofs_list,
        actuator_type=actuator_type,
        kp=position_gain,
        neutral_input=neutral_pose,
    )

    # Add leg adhesion
    fly.add_leg_adhesion()

    # Add visuals
    fly.colorize()
    cam = fly.add_tracking_camera()

    # Create a world and spawn the fly
    world = FlatGroundWorld()
    world.add_fly(fly, spawn_position, spawn_rotation)
    
    return fly, world, cam


fly, world, cam = make_model()

Now we are ready to create a simulation instance. Unlike previous tutorials, here we will use the `GPUSimulation` class from the `warp` module of FlyGym. The key difference is that we can specify how many worlds we want to simulate in parallel (recall that GPUs excell at TODO). Here we will simulate 100 worlds in parallel.

Note that all worlds must be the same model. They can run different _data_ (i.e., what the fly does), but the _models_ themselves must be identical (i.e., flies in all worlds must have the same joint properties, etc.).

In [ ]:
from flygym.warp import GPUSimulation

n_worlds = 100

sim = GPUSimulation(world, n_worlds)

As before, we can attach a renderer to the simulation object. The difference here is that we can specify which worlds to render. If the `worlds` argument is left unspecified, all worlds will be rendered. This, however, might not be desirable because they may consume an enormous amount of memory, depending on how many worlds you are simulating in parallel. Here we will render the first 9 worlds.

In [ ]:
renderer = sim.set_renderer(
    cam,
    playback_speed=0.2,
    output_fps=25,
    worlds=list(range(9)),  # render the first 9 worlds only
)

As in _Tutorial 2: Replaying experimental recordings and inferring dynamical quantities_, we now load some experimentally recorded kinematic data.

In [ ]:
from flygym_examples.spotlight_data import MotionSnippet

snippet = MotionSnippet()
sim_timestep = 1e-4

dof_angles = snippet.get_joint_angles(
    output_timestep=sim_timestep,
    output_dof_order=fly.get_actuated_jointdofs_order(ActuatorType.POSITION),
)
# dof_angles shape: (n_total_steps, n_dofs) — smoothed, interpolated, reordered
print(f"Total steps available: {dof_angles.shape[0]}, n_dofs: {dof_angles.shape[1]}")

Since GPUs achieve their efficiency through parallelization, it is often desirable to break the computing task into small chunks that can be distributed over many GPU threads. For demonstration, we split the kinematic recording into 0.1 _s_ snippets. Each world would get a different snippet (although for any practical application, 0.1 _s_ is probably too short). When all possible 1 _s_ chunks are exhausted, remaining worlds will get repeated copies.

In [ ]:
import numpy as np

sim_seconds = 0.1
sim_steps = int(sim_seconds / sim_timestep)


def make_target_angles_all_worlds(n_worlds, sim_steps=sim_steps):
    """Prepare the input array for all worlds.
    World 0 gets 0s-0.1s, world 1 gets 0.1s-0.2s, etc.
    """
    n_dofs = dof_angles.shape[1]
    dof_angles_all_worlds = np.zeros((n_worlds, sim_steps, n_dofs), dtype=np.float32)
    n_partitions = dof_angles.shape[0] // sim_steps
    for world_id in range(n_worlds):
        partition_idx = world_id % n_partitions
        start_idx = partition_idx * sim_steps
        end_idx = start_idx + sim_steps
        dof_angles_all_worlds[world_id, :, ...] = dof_angles[start_idx:end_idx, ...]
    return dof_angles_all_worlds


dof_angles_all_worlds = make_target_angles_all_worlds(n_worlds)

We can now implement the main simulation loop. Again, this is very similar to the CPU-based simulation loop in _Tutorial 2_. The key difference here is that the `set_leg_adhesion_states` and `set_actuator_inputs` methods require inputs for _all_ worlds. In other words, there is an additional world dimension in the input arrays. This also applies to other methods such as `get_joint_angles`.

In [ ]:
from tqdm import trange  # for progress bar

fly_name = fly.name

sim.reset()

# Turn adhesion on for all 6 legs across all worlds
sim.set_leg_adhesion_states(fly_name, np.ones((n_worlds, 6), dtype=np.float32))

# As before, warm up the simulation for a few steps before applying the control inputs
sim.warmup()

# Run simulation loop
for step in trange(sim_steps):
    control_inputs = dof_angles_all_worlds[:, step, :]
    sim.set_actuator_inputs(fly_name, ActuatorType.POSITION, control_inputs)
    sim.step_with_profile()
    sim.render_as_needed_with_profile()

We can use the `print_performance_report` method to see a breakdown of compute time:

In [ ]:
sim.print_performance_report()

You might have observed two things from the two code blocks above: first, the simulation actually took much longer to run than the CPU version in _Tutorial 2_. Second, it took a long time before the actual simulation—indicated by the progress bar—started, and a bunch of log messages about GPU-side initialization got printed.

The answer to the first question is that we have to remember that we are running 100 worlds in parallel! Therefore, the overall execution speed is 100x faster. (You might notice that even so, this speed is still not very impressive. We will get to faster stuff later on in this tutorial.)

The answer to the second question is that GPU code needs to be compiled first. Therefore, the first time you simulate every model (and the first time only), it's going to take a bit longer.

---

As before, we can display the rendered videos. We will display the nine worlds that we rendered on a grid. This can be done simply by passing a list of world IDs to the `show_in_notebook` method (or `save_video`). If a single integer world ID is given, only that one world will be displayed or saved.

The `scale` argument control the factor by which each video is downscaled: for example, with 0.5, the width and height of each video is reduced by half. This prevents excessive memory usage when many worlds are rendered. If scale is unspecified, the videos will be scaled down so that the overall video has roughly the size of a single camera—this is likely too small though.

In [ ]:
renderer.show_in_notebook(world_id=list(range(9)), scale=0.5)

Instead of 100 worlds, what if we simulate 1000 worlds in parallel?

Let's first consolidate the code in the few blocks above into a `run_simulation` function. In order to make sure that different simulations do not influence each other unexpectedly, we will create the model and generate the target angles from scratch for every simulation. Keep this in mind for the rest of this tutorial: every time we run a simulation, ~2-3 seconds are spent regenerating the model and the data, which is unnecessary for most practical use cases so the overhead is actually smaller.

In [ ]:
def run_simulation(n_worlds):
    dof_angles_all_worlds = make_target_angles_all_worlds(n_worlds)
    
    fly, world, cam = make_model()
    fly_name = fly.name
    
    sim = GPUSimulation(world, n_worlds)
    renderer = sim.set_renderer(
        cam,
        playback_speed=0.2,
        output_fps=25,
        worlds=list(range(9)),  # render the first 9 worlds only
    )
    
    sim.set_leg_adhesion_states(fly_name, np.ones((n_worlds, 6), dtype=np.float32))
    sim.warmup()
    for step in trange(sim_steps):
        control_inputs = dof_angles_all_worlds[:, step, :]
        sim.set_actuator_inputs(fly_name, ActuatorType.POSITION, control_inputs)
        sim.step_with_profile()
        sim.render_as_needed_with_profile()
    
    sim.print_performance_report()
    renderer.show_in_notebook(world_id=list(range(9)), scale=0.5)

We can now simulate 1000 worlds at once (still rendering only 9).

In [ ]:
run_simulation(n_worlds=1000)

Did the simulation take 10x longer? No—in fact, upon closer inspection, the time spent on physics simulation remaind largely constant!

This is the advantage of GPU simulation: TODO. Depending on the hardware and the complexity of the simulation, the optimal number of worlds to simulate in parallel can range from a few hundreds to tens of thousands.

## GPU-side batch rendering

Despite now simulating 1000 worlds at once, we are still rendering only 9 of them. This is becasue rendering each world takes quite a bit of time. Just as how we simulated all 1000 worlds in parallel, can we also do the rendering in parallel on the GPU?

The answer is yes: GPU-side rendering is supported by MuJoCo Warp and Warp itself. To use this, we need to toggle the `use_gpu_batch_rendering` to True.

In [ ]:
def run_simulation_with_batch_rendering(n_worlds):
    dof_angles_all_worlds = make_target_angles_all_worlds(n_worlds)
    
    fly, world, cam = make_model()
    fly_name = fly.name
    
    sim = GPUSimulation(world, n_worlds)
    renderer = sim.set_renderer(
        cam,
        playback_speed=0.2,
        output_fps=25,
        use_gpu_batch_rendering=True,  # enable GPU batch rendering
        worlds=None,  # None: by default, all worlds are rendered
    )
    
    sim.set_leg_adhesion_states(fly_name, np.ones((n_worlds, 6), dtype=np.float32))
    sim.warmup()
    for step in trange(sim_steps):
        control_inputs = dof_angles_all_worlds[:, step, :]
        sim.set_actuator_inputs(fly_name, ActuatorType.POSITION, control_inputs)
        sim.step_with_profile()
        sim.render_as_needed_with_profile()
    
    sim.print_performance_report()
    # still only display the first 9 worlds in the notebook
    renderer.show_in_notebook(world_id=list(range(9)), scale=0.5)

In [ ]:
run_simulation_with_batch_rendering(n_worlds=100)

Note that the appearance of the rendered scene is a bit different. For example, for memory efficiency, we have disabled texture rendering on the fly. The lighting also differs.

On the brighter side, the amount of time spent on rendering has significantly reduced. Don't forget that we are also rendering 1000 worlds now, not just 9, even though we have only displayed the first 9.

## Writing GPU code: Kernel programing and tiling

TODO

## Setting control inputs on the GPU

TODO

In [ ]:
import warp as wp

@wp.kernel
def update_target_angles_kernel(
    dof_angles_all_worlds_gpu: wp.array3d(dtype=wp.float32),  # type: ignore
    step_counter_gpu: wp.array(dtype=wp.int32),  # type: ignore
    curr_target_angles_gpu: wp.array2d(dtype=wp.float32),  # type: ignore
):
    world_id, actuator_id = wp.tid()
    step = step_counter_gpu[0]
    my_target = dof_angles_all_worlds_gpu[world_id, step, actuator_id]
    curr_target_angles_gpu[world_id, actuator_id] = my_target

TODO

In [ ]:
@wp.kernel
def increment_counter_kernel(
    step_counter_gpu: wp.array(dtype=wp.int32),  # type: ignore
):
    step_counter_gpu[0] = step_counter_gpu[0] + 1

TODO.

Pay special attention to the `with ScopedCapture() as advance_sim_capture` block: TODO.

Because TODO, we cannot use `step_with_profile` and `render_as_needed_with_profile` to generate performance statistics: the whole ideas is to do everything on the GPU, so we don't want to have to synchronize GPU-side code with the CPU just for timing. Instead, we will report the overal amount of time taken to run the entire simulation loop.

In [ ]:
from time import perf_counter_ns


def run_simulation_fullgpu(n_worlds, enable_rendering):
    sim_steps = 500  # for timing, run shorter sim to ensure it fits in GPU memory
    dof_angles_all_worlds = make_target_angles_all_worlds(n_worlds, sim_steps)
    n_dofs = dof_angles_all_worlds.shape[-1]

    fly, world, cam = make_model()
    fly_name = fly.name
    sim = GPUSimulation(world, n_worlds)

    if enable_rendering:
        renderer = sim.set_renderer(
            cam,
            playback_speed=0.2,
            output_fps=25,
            use_gpu_batch_rendering=True,
        )

    # Turn adhesion on for all 6 legs across all worlds
    sim.set_leg_adhesion_states(fly_name, np.ones((n_worlds, 6), dtype=np.float32))
    sim.warmup()

    dof_angles_all_worlds_gpu = wp.array(dof_angles_all_worlds)
    curr_target_angles_gpu = wp.zeros((n_worlds, n_dofs), dtype=wp.float32)
    step_counter = wp.array([0], dtype=wp.int32)

    with wp.ScopedCapture() as advance_sim_capture:
        wp.launch(
            update_target_angles_kernel,
            dim=(n_worlds, n_dofs),
            inputs=[dof_angles_all_worlds_gpu, step_counter],
            outputs=[curr_target_angles_gpu],
        )
        sim.set_actuator_inputs(fly_name, ActuatorType.POSITION, curr_target_angles_gpu)
        sim.step()
        wp.launch(increment_counter_kernel, dim=1, outputs=[step_counter])

    start_time = perf_counter_ns()
    for step in trange(sim_steps):
        wp.capture_launch(advance_sim_capture.graph)
        if enable_rendering:
            sim.render_as_needed()
    end_time = perf_counter_ns()
    walltime_seconds = (end_time - start_time) / 1e9
    overall_throughput = n_worlds * sim_steps / walltime_seconds  # steps per sec
    overall_realtime_factor = overall_throughput * sim_timestep  # x realtime

    print(
        f"Simulated {sim_steps} steps * {n_worlds} worlds in {walltime_seconds:.2f}s\n"
        f"Overall throughput: {overall_throughput:.2f} steps/s "
        f"({overall_realtime_factor:.2f}x realtime)"
    )

Let's measure the throughput again. For this benchmark, we will only 500 steps to make sure we have enough memory to render all worlds.

In [ ]:
run_simulation_fullgpu(n_worlds=1000, enable_rendering=True)

Much faster! What if we skip the rendering and focus on the physics simulation itself (e.g. to train controllers using deep reinforcement learning)?

In [ ]:
run_simulation_fullgpu(n_worlds=1000, enable_rendering=False)

Even faster!

We have performed a benchmark on different GPUs. Details and code can be found [here](TODO), and the results are summarized below:

TODO